# WiLoR Remote Inference Server (Colab GPU)

Faster, simpler alternative to the HaMeR server. WiLoR-mini reconstructs MANO with a
lightweight ViT (~10 ms on a T4 vs ~180 ms for HaMeR ViT-H) and is pip-installable with
automatic model download -- no Google Drive checkpoint needed.

Run this on Colab with a GPU runtime. Copy the printed `trycloudflare` URL and pass it to
the local client:

```
python apps/graphgrasp-live/live_retarget.py --ckpt <run20.pt> --source wilor --url https://xxxx.trycloudflare.com
```

Output contract (same shape the client expects):
- `GET  /ping`  -> `{"status": "ok"}`
- `POST /infer` (multipart `frame` jpg) -> `{"keypoints": [[x,y,z]*21], "is_right": 0|1}` for the
  largest detected hand, or `{}` if none. Keypoints are in OpenPose/MediaPipe 21-joint order.

In [ ]:
# Cell 1: install dependencies
!pip install -q fastapi uvicorn nest-asyncio python-multipart
# WiLoR-mini setup.py does `import pip` (needs no build isolation) and its YOLO ckpt needs dill
!pip install -q wheel
!pip install -q --no-build-isolation "git+https://github.com/warmshao/WiLoR-mini"
!pip install -q dill
# cloudflared (free tunnel, no account)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# Cell 2: load WiLoR pipeline (downloads model on first run) + warmup
import torch, numpy as np
from wilor_mini.pipelines.wilor_hand_pose3d_estimation_pipeline import WiLorHandPose3dEstimationPipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32
print("device:", device, "dtype:", dtype)

# wilor_mini defaults focal_length=5000, way off any real webcam -> pred_cam_t_full.z
# (tz = 2*focal_length/bbox_size) comes out in meaningless units, not meters.
# Verified 2026-07-02 against HOGraspNet ground truth (real Kinect fx=942px):
# default focal -> r=0.981 correlation with real depth but arbitrary units (RMSE n/a);
# real focal passed in -> same r=0.981 AND units land in real meters (RMSE 5cm raw,
# residual is a near-constant offset, not a scale error -- see wilor_source.py).
#
# Dell Inspiron 15 3511 built-in cam: 1280x720, diagonal FOV 78.6 deg (estimated from
# spec sheet, NOT measured -- refine with a known-size-object photo once available).
#   diag_px = sqrt(1280**2 + 720**2) = 1468.6
#   fx_px   = (diag_px/2) / tan(78.6deg/2) = 897 px  (assumes fx==fy, no lens distortion)
#   focal_length arg = fx_px * 256 / max(1280,720) = 179.4  (wilor_mini's own 256/img_size rescale)
FOCAL_LENGTH_ARG = 179.4  # TODO: replace with measured value (see docs/estado_pendientes_casa)

pipe = WiLorHandPose3dEstimationPipeline(device=device, dtype=dtype, verbose=False,
                                          focal_length=FOCAL_LENGTH_ARG)
_ = pipe.predict(np.zeros((480, 640, 3), dtype=np.uint8))  # warmup
print("WiLoR ready. focal_length =", FOCAL_LENGTH_ARG)

In [ ]:
# Cell 3: FastAPI server + cloudflared tunnel
import cv2, numpy as np, time, threading, subprocess, re
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
import uvicorn, nest_asyncio
nest_asyncio.apply()

app = FastAPI()

@app.get("/ping")
def ping():
    return {"status": "ok"}

def _largest_hand(outs):
    best, best_area = None, -1.0
    for o in outs:
        x1, y1, x2, y2 = o["hand_bbox"]
        area = abs((x2 - x1) * (y2 - y1))
        if area > best_area:
            best_area, best = area, o
    return best

@app.post("/infer")
async def infer(frame: UploadFile = File(...)):
    data = await frame.read()
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        return JSONResponse({}, status_code=400)
    outs = pipe.predict(img)
    if not outs:
        return JSONResponse({})
    o = _largest_hand(outs)
    wp = o["wilor_preds"]
    kp3d = np.asarray(wp["pred_keypoints_3d"][0], dtype=float)  # (21, 3), OpenPose order, root-relative
    # Wrist 6D extras (stage 2/3 of the sim teleop wrist channel). cam_t = global
    # camera-frame translation, tz from apparent hand size + regressed MANO shape.
    # With FOCAL_LENGTH_ARG set (cell 2, real camera focal instead of wilor_mini's
    # default 5000), verified against HOGraspNet ground truth: r=0.981 correlation,
    # units land in real meters with a near-constant residual offset (~5cm raw) --
    # that residual gets calibrated client-side in wilor_source.py, not here.
    # global_orient = MANO native global rotation (axis-angle). Both .get-guarded so
    # old WiLoR builds that omit them degrade to zeros instead of 500.
    cam_t = np.asarray(wp.get("pred_cam_t_full", [[0.0, 0.0, 0.0]]), dtype=float).reshape(-1)[:3]
    gor = np.asarray(wp.get("global_orient", [[0.0, 0.0, 0.0]]), dtype=float).reshape(-1)[:3]
    return {
        "keypoints": kp3d.tolist(),
        "is_right": int(round(float(o["is_right"]))),
        "cam_t": cam_t.tolist(),
        "global_orient": gor.tolist(),
    }

def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=_run, daemon=True).start()
time.sleep(2)

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        print("\n>>> TUNNEL URL:", m.group(0))
        break
